# Data Collection and Cleaning
Clean NPS visitation panel 

## Objective

This notebook prepares monthly National Park Service visitation data for a causal analysis evaluating the impact of timed-entry reservations at Rocky Mountain National Park.

Raw monthly recreation visits are transformed into a park-month panel dataset suitable for:
- exploratory analysis
- synthetic control modeling
- robustness checks

Source:
National Park Service Visitor Use Statistics

Data: https://catalog.data.gov/dataset/nps-visitor-use-statistics-data-package-2025


In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# plotting style
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.5,
    'figure.figsize' : [10.0, 3.0],
    'figure.dpi' :300,
    'image.aspect': 7 
})

In [2]:

DATA_PATH = '../data/'
FIGURE_PATH = '../figures/'

REC_VISITS_CODE = "TRV"             # Total Recreation Visits

RAW_FILE =  DATA_PATH+'raw/Main_Data.xls'
PARKS_FILE =  DATA_PATH+'raw/national_parks_lookup.csv'


In [3]:
visits = pd.read_csv(RAW_FILE, usecols=["UnitCode", "Year", "Month", "Statistic", "Value"])

# Keep only our parks and only recreation visits.
visits = visits[(visits["Statistic"] == REC_VISITS_CODE)].copy()

# Name of parks given the unit code 
parks = pd.read_csv(PARKS_FILE)

In [12]:
df = visits.merge(parks, on="UnitCode", how="inner")

df = df.rename(columns={"UnitCode": "park_code", "ParkName": "park_name", "Year": "year", 
                        "Month": "month", "Statistic": "statistic", "Value": "visits"})

df = df[[
        "park_name",
        "park_code",
        "year",
        "month",
        "visits"]]

In [13]:
df

,park_name,park_code,year,month,visits
0,Pinnacles National Park,PINN,2007,1,10006
1,Pinnacles National Park,PINN,2007,2,11119
2,Pinnacles National Park,PINN,2007,3,17662
3,Pinnacles National Park,PINN,2007,4,30226
4,Pinnacles National Park,PINN,2007,5,15504
...,...,...,...,...,...
34398,Kenai Fjords National Park,KEFJ,2025,12,7
34399,Kenai Fjords National Park,KEFJ,2025,6,103155
34400,Kenai Fjords National Park,KEFJ,2025,7,132250
34401,Kenai Fjords National Park,KEFJ,2025,10,3828


In [14]:
print (df["park_name"].nunique(), "reporting national parks")

62 reporting national parks


In [15]:
expected_parks = [
    "ACAD","ARCH","BADL","BIBE","BISC","BLCA","BRCA","CANY",
    "CARE","CAVE","CHIS","CONG","CRLA","CUVA","DENA","DEVA",
    "DRTO","EVER","GAAR","JEFF","GLAC","GLBA","GRBA","GRCA",
    "GRSA","GRSM","GRTE","GUMO","HALE","HAVO","HOSP","INDU",
    "ISRO","JOTR","KATM","KEFJ","KICA","KOVA","LACL","LAVO",
    "MACA","MEVE","MORA","NPSA","NEPE","NOCA","OLYM","PEFO",
    "PINN","REDW","ROMO","SAGU","SEKI","SHEN","THRO","VIIS",
    "VOYA","WHSA","WICA","WRST","YELL","YOSE","ZION"
]

missing = set(expected_parks) - set(df["park_code"].unique())

missing

{'SEKI'}

Note: The National Park Service visitation dataset reports Sequoia and Kings Canyon National Parks as a combined unit. Therefore, this analysis contains 62 park reporting units rather than 63 individually designated national parks.

In [16]:
# Lets save this file 

filename = 'monthly_visitation.csv'
output_path = DATA_PATH+'/processed/'+filename

df.to_csv(output_path, index=False)